In [62]:
# 1.1 Import the trading instance from the login notebook
import import_ipynb
from betfair_login import trading

print("Token:", trading.session_token)

# 1.2 Import the libraries we need
import requests
import pandas as pd
import os
import json
import bz2
import glob

# 1.3 Check we have an active Betfair session
if not trading.session_token:
    raise Exception("No active Betfair session. Please log in again.")

print("Successfully connected to Betfair!")

Token: AHYO5q8PsgeTOFanxRDFftf6Tk/kdyVib+DDDns4oEU=
Successfully connected to Betfair!


In [66]:
# Firstly, import the necessary libraries

import betfairlightweight
from betfairlightweight import filters
import pandas as pd
import numpy as np
import os
import datetime
import json
import requests
import json
import bz2
import glob

# Then set up your Betfair API credentials and create a trading instance

with open('credentials.json') as f:
    creds = json.load(f)
    my_username = creds['username']
    my_password = creds['password']
    my_app_key = creds['app_key']


# Create a trading instance and log in interactively
trading = betfairlightweight.APIClient(my_username, my_password, app_key=my_app_key)

trading.login_interactive()
trading.session_token #Prints the session token to the console 

'udSZXmu68PMUwvawoCGFKaVlA37ASsp/GgUpiI7vNnc='

In [86]:
# 2.1 Define the Historical Data API URL
historical_url = "https://historicdata.betfair.com/api/"

# 2.2 Create authentication headers
headers = {
    "ssoid": trading.session_token
}

In [87]:
# 3.1 Define the GetMyData endpoint
get_my_data_url = historical_url + "GetMyData"

# 3.2 Send a GET request to Betfair
response = requests.get(
    get_my_data_url,
    headers=headers
)

# 3.3 Check that the request was successful
response.raise_for_status()

# 3.4 Convert the JSON response into a Pandas DataFrame
my_historical_data = pd.DataFrame(response.json())

# 3.5 Display our purchased historical data
my_historical_data

,sport,plan,forDate,purchaseItemId
0,Other Sports,Basic Plan,2023-01-01T00:00:00,291470
1,Other Sports,Basic Plan,2023-02-01T00:00:00,291470
2,Other Sports,Basic Plan,2023-03-01T00:00:00,291470
3,Other Sports,Basic Plan,2023-04-01T00:00:00,291470
4,Other Sports,Basic Plan,2023-05-01T00:00:00,291470
5,Other Sports,Basic Plan,2023-06-01T00:00:00,291470
6,Other Sports,Basic Plan,2023-07-01T00:00:00,291470
7,Other Sports,Basic Plan,2023-08-01T00:00:00,291470
8,Other Sports,Basic Plan,2023-09-01T00:00:00,291470
9,Other Sports,Basic Plan,2023-10-01T00:00:00,291470


In [88]:
# 4.1 Identify the sport and subscription plan
sport = "Soccer"
plan = "Basic Plan"

# 4.2 Check the months we have purchased
my_historical_data[["sport", "plan", "forDate"]]

,sport,plan,forDate
0,Other Sports,Basic Plan,2023-01-01T00:00:00
1,Other Sports,Basic Plan,2023-02-01T00:00:00
2,Other Sports,Basic Plan,2023-03-01T00:00:00
3,Other Sports,Basic Plan,2023-04-01T00:00:00
4,Other Sports,Basic Plan,2023-05-01T00:00:00
5,Other Sports,Basic Plan,2023-06-01T00:00:00
6,Other Sports,Basic Plan,2023-07-01T00:00:00
7,Other Sports,Basic Plan,2023-08-01T00:00:00
8,Other Sports,Basic Plan,2023-09-01T00:00:00
9,Other Sports,Basic Plan,2023-10-01T00:00:00


In [89]:
# 5.1 Set up our search filters. This will be used to find the specific data we want to download.
filter_options = {
    "sport": sport,
    "plan": plan,
    "fromDay": 1,
    "fromMonth": 2,
    "fromYear": 2026,
    "toDay": 28,
    "toMonth": 2,
    "toYear": 2026,
    "eventId": None,
    "eventName": None,
    "marketTypesCollection": [],
    "countriesCollection": [],
    "fileTypeCollection": ["M"]
}

# 5.2 Send our request to Betfair
response = requests.post(
    historical_url + "GetCollectionOptions",
    headers=headers,
    json=filter_options
)

# 5.3 Check the response
response.raise_for_status()

# 5.4 Convert the response into Python data
options = response.json()

In [90]:
# 6.1 Convert available market types into a DataFrame
market_types = pd.DataFrame(
    options["marketTypesCollection"]
)

# 6.2 Sort by number of available files
market_types = market_types.sort_values(
    "count",
    ascending=False
)

# 6.3 Display the results
market_types

# We will want to lok at MATCH ODDS mainly

,name,count
16,FIRST_HALF_GOALS_25,5121
10,OVER_UNDER_55,5119
7,OVER_UNDER_25,5119
17,ASIAN_HANDICAP,5118
4,MATCH_ODDS,5118
...,...,...
49,ET_OU_GOALS_25,29
48,ET_OU_GOALS_15,29
68,TOTAL_GOALS,25
69,COMBINED_TOTAL,10


In [93]:
team = "Man Utd"
month = 10

In [94]:
# Here we find the historical data for the month of February 2026
# for the sport "Other Sports" and the plan "Basic Plan". 
# We will filter for the event name "Ireland" and the market type "MATCH_ODDS".

# 7.1 Define our download filters
download_filter = {
    "sport": sport,
    "plan": plan,
    "fromDay": 1,
    "fromMonth": month,
    "fromYear": 2026,
    "toDay": 31,
    "toMonth": month,
    "toYear": 2026,
    "eventId": None,
    "eventName": team,
    "marketTypesCollection": ["MATCH_ODDS"],
    "countriesCollection": [],
    "fileTypeCollection": ["M"]
}

# We will be able to change this so the above works dynamically based on the market types we have available.

# 7.2 Request the available historical file paths
response = requests.post(
    historical_url + "DownloadListOfFiles",
    headers=headers,
    json=download_filter
)

# 7.3 Check the response
response.raise_for_status()

# 7.4 Store the file paths
february_files = response.json()

# 7.5 Display the number of files found
print("February files found:", len(february_files))

# 7.6 Display the file paths
february_files

# We can simply repeat this for march by changing the date filters. 

February files found: 13


['/xds_nfs/edp_processed/BASIC/2026/Oct/13/36057653/1.262288793.bz2',
 '/xds_nfs/edp_processed/BASIC/2026/Oct/10/36065343/1.262352985.bz2',
 '/xds_nfs/edp_processed/BASIC/2026/Oct/18/36092108/1.262649738.bz2',
 '/xds_nfs/edp_processed/BASIC/2026/Oct/3/36124764/1.263012156.bz2',
 '/xds_nfs/edp_processed/BASIC/2026/Oct/18/36164668/1.263521767.bz2',
 '/xds_nfs/edp_processed/BASIC/2026/Oct/25/36177347/1.263659373.bz2',
 '/xds_nfs/edp_processed/BASIC/2026/Oct/13/36184355/1.263764522.bz2',
 '/xds_nfs/edp_processed/BASIC/2026/Oct/21/36194583/1.263913210.bz2',
 '/xds_nfs/edp_processed/BASIC/2026/Oct/16/36196704/1.263943633.bz2',
 '/xds_nfs/edp_processed/BASIC/2026/Oct/25/36211698/1.264120264.bz2',
 '/xds_nfs/edp_processed/BASIC/2026/Oct/21/36216654/1.264225855.bz2',
 '/xds_nfs/edp_processed/BASIC/2026/Oct/21/36216833/1.264212622.bz2',
 '/xds_nfs/edp_processed/BASIC/2026/Oct/28/36246687/1.264628613.bz2']

In [100]:
team = "Liverpool"

In [101]:

import requests
import bz2
import json
import pandas as pd
from urllib.parse import urljoin

# ==========================================
# 1. SETTINGS - CHANGE THESE VARIABLES
# ==========================================


# ==========================================
# 2. DOWNLOAD AND EXTRACT EVENT DETAILS
# ==========================================

event_details = []

for file_path in february_files:

    response = requests.get(
        urljoin(historical_url, "DownloadFile"),
        headers=headers,
        params={"filePath": file_path}
    )

    response.raise_for_status()

    decompressed = bz2.decompress(response.content)

    seen_markets = set()

    for line in decompressed.decode("utf-8").splitlines():

        if not line.strip():
            continue

        data = json.loads(line)

        for market in data.get("mc", []):

            market_definition = market.get("marketDefinition", {})

            if not market_definition:
                continue

            event_name = market_definition.get("eventName")
            market_id = market.get("id")
            market_name = market_definition.get("name")
            market_time = market_definition.get("marketTime")

            if not event_name or not market_id:
                continue

            # Only include games involving selected team
            if team.lower() not in event_name.lower():
                continue

            unique_key = market_id

            if unique_key in seen_markets:
                continue

            seen_markets.add(unique_key)

            event_details.append({
                "Fixture": event_name,
                "Market": market_name,
                "Market ID": market_id,
                "Kick-off": market_time,
                "File": file_path.split("/")[-1]
            })


# ==========================================
# 3. CREATE DATAFRAME
# ==========================================

events_df = pd.DataFrame(event_details)

if events_df.empty:
    print(f"No games found for {team}")

else:

    events_df = events_df.drop_duplicates(
        subset=["Market ID"]
    )


    # ==========================================
    # 4. REMOVE YOUTH / WOMEN'S / RESERVE TEAMS
    # ==========================================

    exclude_keywords = [
        "U9", "U10", "U11", "U12", "U13",
        "U14", "U15", "U16", "U17", "U18",
        "U19", "U20", "U21", "U22", "U23",
        "U24", "U25",
        "(W)", "Women", "Women's", "Ladies",
        "Youth", "Reserves", "Reserve"
    ]

    pattern = "|".join(exclude_keywords)

    events_df = events_df[
        ~events_df["Fixture"].str.contains(
            pattern,
            case=False,
            na=False,
            regex=True
        )
    ].copy()


    # ==========================================
    # 5. FORMAT AND SORT
    # ==========================================

    events_df["Kick-off"] = pd.to_datetime(
        events_df["Kick-off"],
        utc=True,
        errors="coerce"
    )

    events_df = events_df.sort_values(
        "Kick-off"
    ).reset_index(drop=True)


    # ==========================================
    # 6. DISPLAY GAMES
    # ==========================================

    games_df = events_df[
        ["Fixture", "Kick-off", "Market ID"]
    ].copy()

    print(f"Team selected: {team}")
    print(f"Total first-team games found: {len(games_df)}")

    display(games_df)


Team selected: Liverpool
Total first-team games found: 0


/var/folders/41/6w1075kn00119ghcyldvzhfw0000gn/T/ipykernel_12967/132445780.py:106: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ~events_df["Fixture"].str.contains(


,Fixture,Kick-off,Market ID


In [95]:
# Here we download the historical data in a dedicated folder. 

# 8.1 Create our historical data folder
data_folder = "man utd games"

os.makedirs(data_folder, exist_ok=True)

# 8.2 Download each historical file
for file_path in february_files:

    # Extract the filename
    filename = os.path.basename(file_path)

    # Download the historical file
    response = requests.get(
        historical_url + "DownloadFile",
        headers=headers,
        params={"filePath": file_path}
    )

    response.raise_for_status()

    # Save the compressed file
    with open(os.path.join(data_folder, filename), "wb") as f:
        f.write(response.content)

print("February downloads complete!")

February downloads complete!


In [75]:
# Repeat the above steps for March 2026. 

download_filter = {
    "sport": sport,
    "plan": plan,
    "fromDay": 1,
    "fromMonth": 3,
    "fromYear": 2026,
    "toDay": 31,
    "toMonth": 3,
    "toYear": 2026,
    "eventId": None,
    "eventName": "Ireland",
    "marketTypesCollection": ["MATCH_ODDS"],
    "countriesCollection": [],
    "fileTypeCollection": ["M"]
}

response = requests.post(
    historical_url + "DownloadListOfFiles",
    headers=headers,
    json=download_filter
)

response.raise_for_status()

march_files = response.json()

print("March files found:", len(march_files))

data_folder = "six_nations_data"

os.makedirs(data_folder, exist_ok=True)

for file_path in march_files:

    filename = os.path.basename(file_path)

    response = requests.get(
        historical_url + "DownloadFile",
        headers=headers,
        params={"filePath": file_path}
    )

    response.raise_for_status()

    with open(os.path.join(data_folder, filename), "wb") as f:
        f.write(response.content)

print("March downloads complete!")


March files found: 7
March downloads complete!


In [76]:
# Shows how many files we have downloaded and stored in our dedicated folder.

# 9.1 Find all compressed historical files
files = glob.glob(
    os.path.join(data_folder, "*.bz2")
)

# 9.2 Display the number of files
print("Total historical files:", len(files))

Total historical files: 15


In [77]:
# 9.3 Open the first historical file
with bz2.open(files[0], "rt") as f:

    # Read the first line
    data = json.loads(f.readline())

# 9.4 Display the contents
data

{'op': 'mcm',
 'clk': '4476886828',
 'pt': 1772382790423,
 'mc': [{'id': '1.254672456',
   'marketDefinition': {'bspMarket': False,
    'turnInPlayEnabled': True,
    'persistenceEnabled': True,
    'marketBaseRate': 5.0,
    'eventId': '35315316',
    'eventTypeId': '7522',
    'numberOfWinners': 1,
    'bettingType': 'ODDS',
    'marketType': 'MATCH_ODDS',
    'marketTime': '2026-03-02T19:30:00.000Z',
    'suspendTime': '2026-03-02T19:30:00.000Z',
    'bspReconciled': False,
    'complete': True,
    'inPlay': False,
    'crossMatching': False,
    'runnersVoidable': False,
    'numberOfActiveRunners': 2,
    'betDelay': 0,
    'status': 'OPEN',
    'runners': [{'status': 'ACTIVE',
      'sortPriority': 1,
      'id': 4830860,
      'name': 'Ireland'},
     {'status': 'ACTIVE',
      'sortPriority': 2,
      'id': 4824792,
      'name': 'Luxembourg'}],
    'regulators': ['MR_INT'],
    'countryCode': 'GB',
    'discountAllowed': True,
    'timezone': 'GMT',
    'openDate': '2026-03-0

In [78]:
market_information = []

for file_path in files:

    market_info = get_market_info(file_path)

    if market_info is not None:
        market_information.append(market_info)

markets_df = pd.DataFrame(market_information)

markets_df.head()

ireland_markets = markets_df[
    markets_df["Event"].str.contains(
        "Ireland",
        case=False,
        na=False
    )
]

ireland_markets

NameError: name 'get_market_info' is not defined